# 2. Microsoft Defender for Cloud — Implementation

## Enabling Defender plans

Defender for Cloud free tier gives basic CSPM. Paid plans add workload protection:

```bash
# Enable Defender for Servers Plan 2
az security pricing create -n VirtualMachines --tier Standard \
  --subplan P2

# Enable Defender for SQL
az security pricing create -n SqlServers --tier Standard

# Enable Defender for Storage
az security pricing create -n StorageAccounts --tier Standard \
  --subplan DefenderForStorageV2

# Enable Defender for Containers
az security pricing create -n Containers --tier Standard

# Enable Defender for Key Vault
az security pricing create -n KeyVaults --tier Standard

# Check all plan statuses
az security pricing list -o table
```

### Defender for Servers — P1 vs P2

| Feature | Plan 1 | Plan 2 |
|---------|--------|--------|
| Defender for Endpoint integration | ✅ | ✅ |
| JIT VM access | ❌ | ✅ |
| File integrity monitoring | ❌ | ✅ |
| Adaptive application controls | ❌ | ✅ |
| Agentless scanning | ❌ | ✅ |
| Vulnerability assessment | ❌ | ✅ |
| Cost | ~$5/server/month | ~$15/server/month |

In [ ]:
import json

# Simulate Secure Score with actionable recommendations
RECOMMENDATIONS = [
    {'id': 1,  'title': 'MFA should be enabled on accounts with owner permissions',              'score': 10, 'status': 'unhealthy', 'severity': 'High',   'category': 'Identity'},
    {'id': 2,  'title': 'Storage accounts should restrict network access',                       'score': 8,  'status': 'unhealthy', 'severity': 'High',   'category': 'Networking'},
    {'id': 3,  'title': 'Subnets should be associated with an NSG',                              'score': 6,  'status': 'healthy',   'severity': 'Medium', 'category': 'Networking'},
    {'id': 4,  'title': 'SQL databases should have vulnerability findings resolved',             'score': 4,  'status': 'unhealthy', 'severity': 'High',   'category': 'Data'},
    {'id': 5,  'title': 'VMs should have endpoint protection installed',                         'score': 6,  'status': 'healthy',   'severity': 'High',   'category': 'Compute'},
    {'id': 6,  'title': 'Key Vault should have purge protection enabled',                        'score': 4,  'status': 'healthy',   'severity': 'Medium', 'category': 'Data'},
    {'id': 7,  'title': 'Diagnostic logs in Key Vault should be enabled',                        'score': 3,  'status': 'unhealthy', 'severity': 'Low',    'category': 'Data'},
    {'id': 8,  'title': 'Azure DDoS Protection should be enabled',                               'score': 8,  'status': 'unhealthy', 'severity': 'Medium', 'category': 'Networking'},
    {'id': 9,  'title': 'Management ports of VMs should be protected with JIT',                  'score': 6,  'status': 'unhealthy', 'severity': 'High',   'category': 'Compute'},
    {'id': 10, 'title': 'Container images should have vulnerability findings resolved',          'score': 5,  'status': 'healthy',   'severity': 'High',   'category': 'Compute'},
]

max_score = sum(r['score'] for r in RECOMMENDATIONS)
current = sum(r['score'] for r in RECOMMENDATIONS if r['status'] == 'healthy')
pct = (current / max_score) * 100

print(f'=== Microsoft Defender for Cloud — Secure Score: {current}/{max_score} ({pct:.0f}%) ===\n')

# Group by severity
for severity in ['High', 'Medium', 'Low']:
    unhealthy = [r for r in RECOMMENDATIONS if r['status'] == 'unhealthy' and r['severity'] == severity]
    if unhealthy:
        print(f'🔴 {severity} severity ({len(unhealthy)} findings):')
        for r in unhealthy:
            print(f'   +{r["score"]} pts | {r["title"]}')
        print()

potential = sum(r['score'] for r in RECOMMENDATIONS if r['status'] == 'unhealthy')
print(f'💡 Fixing all findings adds +{potential} points → {current + potential}/{max_score} ({((current + potential) / max_score) * 100:.0f}%)')

## Compliance standards

```bash
# Add a compliance standard
az security regulatory-compliance-standards list -o table

# Add custom standard (based on a policy initiative)
az security regulatory-compliance-standards create \
  --name 'my-custom-standard' \
  --policy-set-definition /subscriptions/.../policySetDefinitions/my-initiative
```

## Multi-cloud: connecting AWS and GCP

```bash
# Connect AWS account
az security security-connector create -g rg-prod -n aws-connector \
  --environment-name AWS \
  --hierarchy-identifier <aws-account-id> \
  --offerings '[{"offeringType": "CspmMonitorAws"}]'

# Connect GCP project
az security security-connector create -g rg-prod -n gcp-connector \
  --environment-name GCP \
  --hierarchy-identifier <gcp-project-number> \
  --offerings '[{"offeringType": "CspmMonitorGcp"}]'
```

## EASM (External Attack Surface Management)

Discovers and maps your **externally-facing** assets (domains, IPs, web apps) from an attacker's perspective. Finds shadow IT, expired certificates, open ports.

```bash
# Create EASM workspace
az easm workspace create -g rg-prod -n easm-contoso --location eastus

# Add seed domain
az easm discovery-group create -g rg-prod --workspace-name easm-contoso \
  -n contoso-discovery --seeds '[{"kind": "domain", "name": "contoso.com"}]'
```

---
## Summary

| Implementation | Key details |
|---------------|-------------|
| **Defender plans** | Enable per workload type. P1 vs P2 for servers. |
| **Secure Score** | Weighted percentage. Prioritize high-severity findings. |
| **Compliance** | MCSB default. Add CIS, NIST, ISO, PCI. Custom standards via policy initiatives. |
| **Multi-cloud** | Security connectors for AWS and GCP. |
| **EASM** | External attack surface discovery. Seed with domains. |

**Next**: [Notebook 3 — Sentinel Implementation](03_sentinel_implementation.ipynb)